# MNIST batch-effect demo

This notebook downloads an MNIST subset with two unequal batches (n=10,000 and n=20,000) and adds a different fixed random pixel-space vector to each batch. The offset is fixed within a batch, so it acts like a technical batch effect rather than sample-specific noise.

In [ ]:
from pathlib import Path
import contextlib
import io
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from torchvision import datasets

SEED = 7
BATCH_SIZES = (10_000, 20_000)
SUBSET_SIZE = sum(BATCH_SIZES)
EFFECT_STRENGTH = 0.55
IMAGE_SHAPE = (28, 28)

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

plt.style.use("seaborn-v0_8-whitegrid")

## Download and split a label-balanced subset

The two batches intentionally have different sample counts, but each batch is balanced across digits. That keeps label proportions matched while testing the larger n=10,000 and n=20,000 batch sizes.

In [ ]:
if any(size % 10 != 0 for size in BATCH_SIZES):
    raise ValueError("Choose batch sizes divisible by 10 for balanced digit splits.")

with contextlib.redirect_stderr(io.StringIO()):
    mnist = datasets.MNIST(root=DATA_DIR, train=True, download=True)
targets = mnist.targets.numpy()

per_digit_by_batch = [size // 10 for size in BATCH_SIZES]

selected_indices = []
batch_ids = []
for digit in range(10):
    digit_indices = np.flatnonzero(targets == digit)
    needed = sum(per_digit_by_batch)
    if len(digit_indices) < needed:
        raise ValueError(
            f"Digit {digit} only has {len(digit_indices)} training images, "
            f"but {needed} are needed for batch sizes {BATCH_SIZES}."
        )

    chosen = rng.choice(digit_indices, size=needed, replace=False)
    rng.shuffle(chosen)

    start = 0
    for batch, per_digit_count in enumerate(per_digit_by_batch):
        stop = start + per_digit_count
        selected_indices.append(chosen[start:stop])
        batch_ids.extend([batch] * per_digit_count)
        start = stop

selected_indices = np.concatenate(selected_indices)
batch_ids = np.asarray(batch_ids, dtype=np.int64)

perm = rng.permutation(SUBSET_SIZE)
selected_indices = selected_indices[perm]
batch_ids = batch_ids[perm]

images = mnist.data[torch.as_tensor(selected_indices)].numpy().astype(np.float32) / 255.0
labels = targets[selected_indices]

counts = np.stack([
    np.bincount(labels[batch_ids == batch], minlength=10)
    for batch in range(len(BATCH_SIZES))
])

print(f"MNIST subset: {len(images):,} images")
print(f"Batch sizes: {np.bincount(batch_ids).tolist()}")
print("Digit counts by batch, rows are batch 0 and batch 1:")
print(counts)
print("Digit proportions by batch:")
print(np.round(counts / counts.sum(axis=1, keepdims=True), 3))

## Add batch-specific random vectors

Both batches get an additive offset, but the vector is different for each batch. Here the offsets are dense random pixel-space vectors, one fixed vector per batch.

In [ ]:
effect_rng = np.random.default_rng(SEED + 1)
batch_effect_vectors = effect_rng.uniform(
    -EFFECT_STRENGTH,
    EFFECT_STRENGTH,
    size=(2, *IMAGE_SHAPE),
).astype(np.float32)

images_with_batch_effect = np.clip(
    images + batch_effect_vectors[batch_ids],
    0.0,
    1.0,
)

flat_original = images.reshape(len(images), -1)
flat_with_batch_effect = images_with_batch_effect.reshape(len(images), -1)


def batch_mean_distance(flat_images):
    mean_0 = flat_images[batch_ids == 0].mean(axis=0)
    mean_1 = flat_images[batch_ids == 1].mean(axis=0)
    return np.linalg.norm(mean_0 - mean_1)


print(f"Batch mean distance before adding vectors: {batch_mean_distance(flat_original):.3f}")
print(f"Batch mean distance after adding vectors:  {batch_mean_distance(flat_with_batch_effect):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3), constrained_layout=True)
for batch, ax in enumerate(axes):
    im = ax.imshow(
        batch_effect_vectors[batch],
        cmap="coolwarm",
        vmin=-EFFECT_STRENGTH,
        vmax=EFFECT_STRENGTH,
    )
    ax.set_title(f"Batch {batch} random vector")
    ax.axis("off")

fig.colorbar(im, ax=axes, shrink=0.8, label="additive pixel offset")
fig.suptitle("The same random vector is added to every image within a batch", y=1.04);

## Example images with the batch effect

Each column shows the same digit class in the original and batch-effected versions. The digit itself is still present, but a batch-specific random offset is now visible.

In [ ]:
digits_to_show = np.arange(8)


def pick_one_per_digit(batch, digits):
    picks = []
    for digit in digits:
        matches = np.flatnonzero((batch_ids == batch) & (labels == digit))
        picks.append(matches[0])
    return np.asarray(picks)


row_specs = [
    (0, images, "Batch 0 original"),
    (0, images_with_batch_effect, "Batch 0 + vector"),
    (1, images, "Batch 1 original"),
    (1, images_with_batch_effect, "Batch 1 + vector"),
]

fig, axes = plt.subplots(
    len(row_specs),
    len(digits_to_show),
    figsize=(1.3 * len(digits_to_show), 5.4),
    constrained_layout=True,
)

for row, (batch, image_array, row_label) in enumerate(row_specs):
    picks = pick_one_per_digit(batch, digits_to_show)
    for col, idx in enumerate(picks):
        ax = axes[row, col]
        ax.imshow(image_array[idx], cmap="gray", vmin=0, vmax=1)
        if row == 0:
            ax.set_title(f"digit {labels[idx]}", fontsize=9)
        if col == 0:
            ax.set_ylabel(row_label, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle("Example MNIST images before and after adding batch-specific vectors", y=1.02);

## Mean images reveal the artificial batch difference

The original batch means are similar because the subset is balanced. After adding the vectors, the mean images encode the batch-specific random offsets.

In [ ]:
mean_original_0 = images[batch_ids == 0].mean(axis=0)
mean_original_1 = images[batch_ids == 1].mean(axis=0)
mean_effect_0 = images_with_batch_effect[batch_ids == 0].mean(axis=0)
mean_effect_1 = images_with_batch_effect[batch_ids == 1].mean(axis=0)

fig, axes = plt.subplots(2, 3, figsize=(8.5, 5.4), constrained_layout=True)

panels = [
    (axes[0, 0], mean_original_0, "Original mean, batch 0", "gray", 0, 1),
    (axes[0, 1], mean_original_1, "Original mean, batch 1", "gray", 0, 1),
    (axes[0, 2], mean_original_0 - mean_original_1, "Original mean difference", "coolwarm", -0.20, 0.20),
    (axes[1, 0], mean_effect_0, "With effect, batch 0", "gray", 0, 1),
    (axes[1, 1], mean_effect_1, "With effect, batch 1", "gray", 0, 1),
    (axes[1, 2], mean_effect_0 - mean_effect_1, "Effect mean difference", "coolwarm", -EFFECT_STRENGTH, EFFECT_STRENGTH),
]

for ax, image, title, cmap, vmin, vmax in panels:
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    if "difference" in title.lower():
        fig.colorbar(im, ax=ax, shrink=0.75)

fig.suptitle("Batch means before and after the random-vector offsets", y=1.02);

## PCA view of the batch effect

A pixel-space PCA makes the effect visible at the population level. Before adding vectors, batches overlap; afterward, the constant offsets pull the batches apart.

In [ ]:
def plot_pca_by_batch(flat_images, title, ax):
    coords = PCA(n_components=2).fit_transform(flat_images)
    colors = ["tab:blue", "tab:orange"]
    for batch, color in zip((0, 1), colors):
        mask = batch_ids == batch
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=8,
            alpha=0.45,
            c=color,
            label=f"batch {batch}",
            linewidths=0,
        )
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend(frameon=False, markerscale=2)
    return coords


fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), constrained_layout=True)
coords_original = plot_pca_by_batch(flat_original, "Original images", axes[0])
coords_effect = plot_pca_by_batch(flat_with_batch_effect, "After adding batch vectors", axes[1])
fig.suptitle("The artificial constant vectors create a visible batch effect", y=1.03);

## Joint t-SNE and OT-tSNE embeddings

The embedding cell uses a larger method subset while preserving the 1:2 batch-size ratio: 100 images per digit from batch 0 and 200 images per digit from batch 1. Joint t-SNE embeds both batches together from the batch-effected pixels, while OT-tSNE builds a graph from the optimal-transport coupling between batches and then runs t-SNE on that graph.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="IProgress not found.*")
    from ne_aligned_utils.methods import (
        compute_ot_tsne_embedding_from_arrays,
        compute_tsne_embedding_from_arrays,
    )

TSNE_PER_DIGIT_BY_BATCH = (100, 200)
TSNE_PCA_COMPONENTS = 50
TSNE_PERPLEXITY = 50.0
TSNE_N_ITER = 750
TSNE_EARLY_EXAGGERATION_ITER = 250
OT_TSNE_EPS_TIMES = 1.0
OT_TSNE_EFFECTIVE_K = 5
OT_TSNE_N_EPOCHS = 750

embedding_indices_by_batch = []
for batch in (0, 1):
    batch_indices = []
    for digit in range(10):
        matches = np.flatnonzero((batch_ids == batch) & (labels == digit))
        sample_size = TSNE_PER_DIGIT_BY_BATCH[batch]
        if matches.size < sample_size:
            raise ValueError(
                f"Batch {batch}, digit {digit} only has {matches.size} images, "
                f"but {sample_size} were requested."
            )
        chosen = rng.choice(matches, size=sample_size, replace=False)
        batch_indices.append(chosen)
    batch_indices = np.concatenate(batch_indices)
    rng.shuffle(batch_indices)
    embedding_indices_by_batch.append(batch_indices)

idx0, idx1 = embedding_indices_by_batch
x0 = images_with_batch_effect[idx0].reshape(len(idx0), -1)
x1 = images_with_batch_effect[idx1].reshape(len(idx1), -1)
embedding_labels = np.concatenate([labels[idx0], labels[idx1]])
embedding_batch_ids = np.concatenate([
    np.zeros(len(idx0), dtype=int),
    np.ones(len(idx1), dtype=int),
])

joint_tsne_embedding, joint_tsne_artifacts = compute_tsne_embedding_from_arrays(
    x0,
    x1,
    random_state=SEED,
    perplexity=TSNE_PERPLEXITY,
    pca_components=TSNE_PCA_COMPONENTS,
    n_iter=TSNE_N_ITER,
    early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
    return_artifacts=True,
)

ot_tsne_embedding, ot_tsne_artifacts = compute_ot_tsne_embedding_from_arrays(
    x0,
    x1,
    eps_times=OT_TSNE_EPS_TIMES,
    effective_k=OT_TSNE_EFFECTIVE_K,
    random_state=SEED,
    n_epochs=OT_TSNE_N_EPOCHS,
    return_artifacts=True,
)

print(f"Embedding subset: {len(embedding_labels):,} images ({len(idx0)} batch 0, {len(idx1)} batch 1)")
print(f"Joint t-SNE PCA components used: {joint_tsne_artifacts['pca_components']}")
print(
    "OT-tSNE: "
    f"eps_times={OT_TSNE_EPS_TIMES:g}, "
    f"effective_k={ot_tsne_artifacts['effective_k']} "
    f"({ot_tsne_artifacts['batch0_k']} within batch 0 + {ot_tsne_artifacts['batch1_k']} within batch 1), "
    f"epsilon={ot_tsne_artifacts['epsilon']:.4g}"
)


def scatter_by_batch(ax, embedding, title):
    colors = np.array(["tab:blue", "tab:orange"])
    order = np.random.default_rng(SEED).permutation(len(embedding_batch_ids))
    for batch in (0, 1):
        mask = embedding_batch_ids[order] == batch
        ax.scatter(
            embedding[order][mask, 0],
            embedding[order][mask, 1],
            s=8,
            alpha=0.65,
            c=colors[batch],
            label=f"batch {batch}",
            linewidths=0,
        )
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(frameon=False, markerscale=2, loc="best")


def scatter_by_digit(ax, embedding, title):
    order = np.random.default_rng(SEED).permutation(len(embedding_labels))
    scatter = ax.scatter(
        embedding[order, 0],
        embedding[order, 1],
        s=8,
        alpha=0.65,
        c=embedding_labels[order],
        cmap="tab10",
        vmin=-0.5,
        vmax=9.5,
        linewidths=0,
    )
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    return scatter


fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
scatter_by_batch(axes[0, 0], joint_tsne_embedding, "Joint t-SNE colored by batch")
scatter_by_batch(axes[0, 1], ot_tsne_embedding, "OT-tSNE colored by batch")
scatter = scatter_by_digit(axes[1, 0], joint_tsne_embedding, "Joint t-SNE colored by digit")
scatter_by_digit(axes[1, 1], ot_tsne_embedding, "OT-tSNE colored by digit")

cbar = fig.colorbar(scatter, ax=axes[1, :], ticks=np.arange(10), shrink=0.85, label="digit")
cbar.ax.set_yticklabels([str(digit) for digit in range(10)])
fig.suptitle("Joint t-SNE versus OT-tSNE on the batch-effected MNIST subset", y=1.02);

## Pure MNIST t-SNE

This cell runs t-SNE on the same larger method subset before adding any batch vectors.

In [ ]:
pure_x0 = images[idx0].reshape(len(idx0), -1)
pure_x1 = images[idx1].reshape(len(idx1), -1)

pure_mnist_tsne_embedding, pure_mnist_tsne_artifacts = compute_tsne_embedding_from_arrays(
    pure_x0,
    pure_x1,
    random_state=SEED,
    perplexity=TSNE_PERPLEXITY,
    pca_components=TSNE_PCA_COMPONENTS,
    n_iter=TSNE_N_ITER,
    early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
    return_artifacts=True,
)

print(
    f"Pure MNIST t-SNE: {len(embedding_labels):,} images, "
    f"PCA components used: {pure_mnist_tsne_artifacts['pca_components']}"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
scatter_by_batch(axes[0], pure_mnist_tsne_embedding, "Pure MNIST t-SNE colored by batch")
scatter = scatter_by_digit(axes[1], pure_mnist_tsne_embedding, "Pure MNIST t-SNE colored by digit")

cbar = fig.colorbar(scatter, ax=axes, ticks=np.arange(10), shrink=0.85, label="digit")
cbar.ax.set_yticklabels([str(digit) for digit in range(10)])
fig.suptitle("t-SNE on original MNIST pixels before adding batch vectors", y=1.03);

In [ ]:
from ne_aligned_utils.methods import compute_tsne_embedding

ONE_BATCH = 0
one_batch_indices = embedding_indices_by_batch[ONE_BATCH]
one_batch_x = images_with_batch_effect[one_batch_indices].reshape(len(one_batch_indices), -1)
one_batch_labels = labels[one_batch_indices]

one_batch_pca_dim = min(TSNE_PCA_COMPONENTS, one_batch_x.shape[0] - 1, one_batch_x.shape[1])
if one_batch_pca_dim >= 2 and one_batch_pca_dim < one_batch_x.shape[1]:
    one_batch_tsne_input = PCA(
        n_components=one_batch_pca_dim,
        random_state=SEED,
    ).fit_transform(one_batch_x)
else:
    one_batch_tsne_input = one_batch_x
    one_batch_pca_dim = None

one_batch_tsne_embedding, one_batch_tsne_artifacts = compute_tsne_embedding(
    one_batch_tsne_input,
    random_state=SEED,
    perplexity=TSNE_PERPLEXITY,
    n_iter=TSNE_N_ITER,
    early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
    return_artifacts=True,
)

print(
    f"One-batch t-SNE: batch {ONE_BATCH}, "
    f"{len(one_batch_labels):,} images, "
    f"perplexity={one_batch_tsne_artifacts['perplexity']:.1f}, "
    f"PCA components used: {one_batch_pca_dim}"
)

order = np.random.default_rng(SEED).permutation(len(one_batch_labels))
fig, ax = plt.subplots(figsize=(5.8, 5.2), constrained_layout=True)
scatter = ax.scatter(
    one_batch_tsne_embedding[order, 0],
    one_batch_tsne_embedding[order, 1],
    s=10,
    alpha=0.7,
    c=one_batch_labels[order],
    cmap="tab10",
    vmin=-0.5,
    vmax=9.5,
    linewidths=0,
)
ax.set_title(f"t-SNE for batch {ONE_BATCH} only, colored by digit")
ax.set_xticks([])
ax.set_yticks([])

cbar = fig.colorbar(scatter, ax=ax, ticks=np.arange(10), shrink=0.85, label="digit")
cbar.ax.set_yticklabels([str(digit) for digit in range(10)]);

## Takeaway

The two batches have different sample counts but matching digit proportions, so the visible separation is introduced by the batch-specific random vectors rather than label composition. This gives a simple synthetic example where an alignment or batch-correction method should preserve digit identity while removing the technical offset.

## Distributed MNIST benchmark

The cells below save the MNIST benchmark subset as a named repo dataset, submit the non-scVI methods through `embeddings_for_benchmarks.py`, aggregate the resulting embeddings, and compute kBET plus silhouette score with the runner's `score-aggregate` command.

In [ ]:
import shlex
import subprocess

import pandas as pd

DEFAULT_METHODS = (
    "eot-t0",
    "ot-tsne",
    "eot-50-tsne",
    "joint-pca",
    "joint-tsne",
    "seurat",
    "harmony-2d",
    "harmony-50-tsne",
)
SCVI_METHODS = ("scvi-2d", "scvi-50-tsne")

MNIST_SKIP_METHODS = set(SCVI_METHODS)
MNIST_BENCHMARK_METHODS = [
    method for method in DEFAULT_METHODS
    if method not in MNIST_SKIP_METHODS
]
MNIST_BENCHMARK_N_SEEDS = 1
MNIST_BENCHMARK_EPS_TIMES = [OT_TSNE_EPS_TIMES]
MNIST_BENCHMARK_EFFECTIVE_KS = [OT_TSNE_EFFECTIVE_K]
MNIST_BENCHMARK_N_EPOCHS = TSNE_N_ITER
MNIST_BENCHMARK_TSNE_PCA_COMPONENTS = TSNE_PCA_COMPONENTS
MNIST_BENCHMARK_PERPLEXITY = TSNE_PERPLEXITY

MNIST_DATASET_DIR = DATA_DIR / "mnist"
MNIST_BENCHMARK_OUTPUT_DIR = DATA_DIR / "benchmark_embeddings" / "mnist"
MNIST_BENCHMARK_SCRIPT = SRC_DIR / "ne_aligned_utils" / "embeddings_for_benchmarks.py"
MNIST_BENCHMARK_CONFIG = (
    "n_per_batch_all"
    f"__q_2"
    f"__epochs_{int(MNIST_BENCHMARK_N_EPOCHS)}"
    f"__tsne_pca_{int(MNIST_BENCHMARK_TSNE_PCA_COMPONENTS)}"
)
MNIST_BENCHMARK_TASK_ROOT = MNIST_BENCHMARK_OUTPUT_DIR / MNIST_BENCHMARK_CONFIG
MNIST_BENCHMARK_AGGREGATE_PATH = MNIST_BENCHMARK_TASK_ROOT / "mnist_benchmark_embeddings.npz"
MNIST_BENCHMARK_METRICS_PATH = MNIST_BENCHMARK_TASK_ROOT / "mnist_benchmark_metrics.csv"

MNIST_DATASET_DIR.mkdir(parents=True, exist_ok=True)
np.save(MNIST_DATASET_DIR / "X1.npy", np.asarray(x0, dtype=np.float32))
np.save(MNIST_DATASET_DIR / "X2.npy", np.asarray(x1, dtype=np.float32))
np.save(MNIST_DATASET_DIR / "labels1.npy", np.asarray(labels[idx0], dtype=int))
np.save(MNIST_DATASET_DIR / "labels2.npy", np.asarray(labels[idx1], dtype=int))
np.save(MNIST_DATASET_DIR / "cell_type_names.npy", np.asarray([str(digit) for digit in range(10)]))
np.save(MNIST_DATASET_DIR / "dataset_names.npy", np.asarray(["MNIST batch 0", "MNIST batch 1"]))

print(f"Saved distributed benchmark input to {MNIST_DATASET_DIR}")
print("Methods:", MNIST_BENCHMARK_METHODS)
MNIST_BENCHMARK_AGGREGATE_PATH

In [ ]:
def shell_join(command):
    return shlex.join([str(part) for part in command])


mnist_common_cli = [
    "--dataset", "mnist",
    "--methods", *MNIST_BENCHMARK_METHODS,
    "--embedding-dim", "2",
    "--n-epochs", str(int(MNIST_BENCHMARK_N_EPOCHS)),
    "--tsne-pca-components", str(int(MNIST_BENCHMARK_TSNE_PCA_COMPONENTS)),
    "--perplexity", f"{float(MNIST_BENCHMARK_PERPLEXITY):g}",
    "--n-jobs", "1",
    "--data-root", str(DATA_DIR),
    "--output-dir", str(MNIST_BENCHMARK_OUTPUT_DIR),
    "--eps-times", *(f"{float(value):g}" for value in MNIST_BENCHMARK_EPS_TIMES),
    "--effective-k", *(str(int(value)) for value in MNIST_BENCHMARK_EFFECTIVE_KS),
]

mnist_submit_command = [
    sys.executable,
    str(MNIST_BENCHMARK_SCRIPT),
    "submit-slurm",
    "--n-seeds", str(int(MNIST_BENCHMARK_N_SEEDS)),
    "--overwrite-manifest",
    *mnist_common_cli,
]
mnist_aggregate_command = [
    sys.executable,
    str(MNIST_BENCHMARK_SCRIPT),
    "aggregate",
    "--n-seeds", str(int(MNIST_BENCHMARK_N_SEEDS)),
    *mnist_common_cli,
]
mnist_score_command = [
    sys.executable,
    str(MNIST_BENCHMARK_SCRIPT),
    "score-aggregate",
    "--aggregate-cache", str(MNIST_BENCHMARK_AGGREGATE_PATH),
    "--output-csv", str(MNIST_BENCHMARK_METRICS_PATH),
    "--methods", *MNIST_BENCHMARK_METHODS,
]

MNIST_SUBMIT_SLURM = False
MNIST_SUBMIT_DRY_RUN = False
if MNIST_SUBMIT_DRY_RUN:
    mnist_submit_command.append("--dry-run")

if MNIST_SUBMIT_SLURM:
    subprocess.run(mnist_submit_command, check=True)
else:
    print("Submit Slurm jobs with:")
    print(shell_join(mnist_submit_command))

print("After the jobs finish, aggregate with:")
print(shell_join(mnist_aggregate_command))
print("Then score the aggregate with:")
print(shell_join(mnist_score_command))

In [ ]:
MNIST_RUN_AGGREGATE = False

if MNIST_RUN_AGGREGATE:
    subprocess.run(mnist_aggregate_command, check=True)

if MNIST_BENCHMARK_AGGREGATE_PATH.exists():
    subprocess.run(mnist_score_command, check=True)
    mnist_method_metrics = pd.read_csv(MNIST_BENCHMARK_METRICS_PATH)
else:
    mnist_method_metrics = None
    print(f"Missing aggregate cache: {MNIST_BENCHMARK_AGGREGATE_PATH}")
    print("Run the aggregate command after the Slurm jobs finish:")
    print(shell_join(mnist_aggregate_command))

mnist_method_metrics

In [ ]:
if mnist_method_metrics is None:
    raise FileNotFoundError(
        "Run the distributed MNIST benchmark, aggregate it, and score it before plotting."
    )

mnist_method_summary = (
    mnist_method_metrics
    .groupby(["method", "method_label"], as_index=False)
    .agg(
        neighborhood_purity_mean=("neighborhood_purity_score", "mean"),
        neighborhood_purity_std=("neighborhood_purity_score", "std"),
        kbet_mean=("kbet_score", "mean"),
        kbet_std=("kbet_score", "std"),
        n_runs=("seed", "nunique"),
    )
    .sort_values(["kbet_mean", "neighborhood_purity_mean"], ascending=False)
)
display(mnist_method_summary.round(3).reset_index(drop=True))

plot_data = mnist_method_summary.dropna(subset=["kbet_mean", "neighborhood_purity_mean"]).copy()

fig, ax = plt.subplots(figsize=(8.4, 6.6), constrained_layout=True)
colors = plt.get_cmap("tab10")(np.linspace(0, 1, len(plot_data)))
for color, (_, row) in zip(colors, plot_data.iterrows()):
    ax.scatter(
        row["kbet_mean"],
        row["neighborhood_purity_mean"],
        s=58,
        color=color,
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )
    ax.annotate(
        row["method_label"],
        (row["kbet_mean"], row["neighborhood_purity_mean"]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=9,
        ha="left",
        va="bottom",
    )

x_values = plot_data["kbet_mean"].to_numpy(dtype=float)
y_values = plot_data["neighborhood_purity_mean"].to_numpy(dtype=float)
x_pad = max(0.02, 0.08 * float(np.nanmax(x_values) - np.nanmin(x_values) or 1.0))
y_pad = max(0.02, 0.08 * float(np.nanmax(y_values) - np.nanmin(y_values) or 1.0))
ax.set_xlim(max(0.0, float(np.nanmin(x_values) - x_pad)), min(1.0, float(np.nanmax(x_values) + x_pad)))
ax.set_ylim(max(0.0, float(np.nanmin(y_values) - y_pad)), min(1.0, float(np.nanmax(y_values) + y_pad)))
ax.set_xlabel("mean kBET score")
ax.set_ylabel("mean neighborhood purity")
ax.set_title("MNIST batch-effect benchmark")
ax.grid(alpha=0.25)
fig;

## Benchmark embedding plots

Plot the aggregated distributed benchmark embeddings, colored by digit and by synthetic batch.

In [ ]:
from pathlib import Path

if "MNIST_BENCHMARK_AGGREGATE_PATH" not in globals():
    notebook_cwd = Path.cwd()
    project_root = notebook_cwd.parent if notebook_cwd.name == "notebooks" else notebook_cwd
    MNIST_BENCHMARK_AGGREGATE_PATH = (
        project_root
        / "data"
        / "benchmark_embeddings"
        / "mnist"
        / "n_per_batch_all__q_2__epochs_750__tsne_pca_50"
        / "mnist_benchmark_embeddings.npz"
    )
    MNIST_BENCHMARK_EPS_TIMES = [1.0]
    MNIST_BENCHMARK_EFFECTIVE_KS = [5]
    SEED = 7

if not MNIST_BENCHMARK_AGGREGATE_PATH.exists():
    raise FileNotFoundError(f"Missing aggregate cache: {MNIST_BENCHMARK_AGGREGATE_PATH}")

with np.load(MNIST_BENCHMARK_AGGREGATE_PATH, allow_pickle=False) as loaded:
    mnist_embedding_cache = {key: loaded[key] for key in loaded.files}

seed_index = 0
eps_values = np.asarray(mnist_embedding_cache.get("eps_times", []), dtype=float)
effective_ks = np.asarray(mnist_embedding_cache.get("effective_ks", []), dtype=int)
target_eps = float(MNIST_BENCHMARK_EPS_TIMES[0]) if "MNIST_BENCHMARK_EPS_TIMES" in globals() else float(eps_values[0])
target_k = int(MNIST_BENCHMARK_EFFECTIVE_KS[0]) if "MNIST_BENCHMARK_EFFECTIVE_KS" in globals() else int(effective_ks[0])


def first_matching_index(values, target, *, is_float=False):
    values = np.asarray(values)
    if values.size == 0:
        return 0
    if is_float:
        matches = np.flatnonzero(np.isclose(values.astype(float), float(target)))
    else:
        matches = np.flatnonzero(values == target)
    return int(matches[0]) if matches.size else 0


eps_index = first_matching_index(eps_values, target_eps, is_float=True)
k_index = first_matching_index(effective_ks, target_k)
mnist_embedding_labels = np.concatenate(
    [
        mnist_embedding_cache["labels0"][seed_index],
        mnist_embedding_cache["labels1"][seed_index],
    ]
).astype(int)
mnist_embedding_batch_ids = mnist_embedding_cache["batch_ids"].astype(int)
mnist_embedding_dataset_names = [str(name) for name in mnist_embedding_cache["dataset_names"]]
mnist_label_names = [str(name) for name in mnist_embedding_cache["label_names"]]
available_methods = {str(method) for method in np.atleast_1d(mnist_embedding_cache.get("methods", []))}
mnist_embedding_records = []


def should_plot_method(method):
    return not available_methods or method in available_methods


if should_plot_method("eot-t0") and "eot_t0_embeddings" in mnist_embedding_cache:
    mnist_embedding_records.append(
        (
            f"EOT t=0 (eps={float(eps_values[eps_index]):g})",
            mnist_embedding_cache["eot_t0_embeddings"][seed_index, eps_index],
        )
    )
if should_plot_method("ot-tsne") and "ot_tsne_embeddings" in mnist_embedding_cache:
    mnist_embedding_records.append(
        (
            f"OT-tSNE (eps={float(eps_values[eps_index]):g}, k={int(effective_ks[k_index])})",
            mnist_embedding_cache["ot_tsne_embeddings"][seed_index, eps_index, k_index],
        )
    )
if should_plot_method("eot-50-tsne") and "eot_50_tsne_embeddings" in mnist_embedding_cache:
    mnist_embedding_records.append(
        (
            f"EOT50 + tSNE (eps={float(eps_values[eps_index]):g})",
            mnist_embedding_cache["eot_50_tsne_embeddings"][seed_index, eps_index],
        )
    )
for method, label, key in [
    ("joint-pca", "Joint PCA", "joint_pca_embeddings"),
    ("joint-tsne", "Joint tSNE", "joint_tsne_embeddings"),
    ("seurat", "Seurat", "seurat_embeddings"),
    ("harmony-2d", "Harmony 2D", "harmony_2d_embeddings"),
    ("harmony-50-tsne", "Harmony50 + tSNE", "harmony_50_tsne_embeddings"),
]:
    if should_plot_method(method) and key in mnist_embedding_cache:
        mnist_embedding_records.append((label, mnist_embedding_cache[key][seed_index]))

if not mnist_embedding_records:
    raise ValueError("No benchmark embeddings were found in the aggregate cache.")

rng = np.random.default_rng(SEED if "SEED" in globals() else 7)
plot_order = rng.permutation(mnist_embedding_labels.size)
batch_palette = np.array(["#4C78A8", "#F58518", "#54A24B", "#E45756"])


def style_embedding_axis(ax, embedding):
    finite = np.isfinite(embedding).all(axis=1)
    points = embedding[finite]
    if points.size:
        lower = np.nanquantile(points, 0.005, axis=0)
        upper = np.nanquantile(points, 0.995, axis=0)
        span = np.maximum(upper - lower, 1e-6)
        pad = 0.08 * span
        ax.set_xlim(lower[0] - pad[0], upper[0] + pad[0])
        ax.set_ylim(lower[1] - pad[1], upper[1] + pad[1])
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


n_methods = len(mnist_embedding_records)
method_columns = 2
method_rows = int(np.ceil(n_methods / method_columns))
plot_columns = method_columns * 2
digit_cmap = plt.get_cmap("tab10", max(10, len(mnist_label_names)))
digit_vmax = max(9.5, len(mnist_label_names) - 0.5)

fig = plt.figure(figsize=(15.5, max(5.8, 2.6 * method_rows)), constrained_layout=True)
grid = fig.add_gridspec(
    method_rows,
    plot_columns + 1,
    width_ratios=[1.0, 1.0] * method_columns + [0.72],
    wspace=0.06,
    hspace=0.16,
)
legend_ax = fig.add_subplot(grid[:, -1])
legend_ax.axis("off")

for method_index, (method_label, embedding) in enumerate(mnist_embedding_records):
    embedding = np.asarray(embedding, dtype=float)
    method_row = method_index // method_columns
    method_column = method_index % method_columns
    digit_ax = fig.add_subplot(grid[method_row, 2 * method_column])
    batch_ax = fig.add_subplot(grid[method_row, 2 * method_column + 1])
    digit_ax.scatter(
        embedding[plot_order, 0],
        embedding[plot_order, 1],
        c=mnist_embedding_labels[plot_order],
        cmap=digit_cmap,
        vmin=-0.5,
        vmax=digit_vmax,
        s=5,
        alpha=0.78,
        linewidths=0,
        rasterized=True,
    )
    for batch_id, batch_name in enumerate(mnist_embedding_dataset_names):
        batch_mask = mnist_embedding_batch_ids == batch_id
        batch_order = plot_order[batch_mask[plot_order]]
        batch_ax.scatter(
            embedding[batch_order, 0],
            embedding[batch_order, 1],
            color=batch_palette[batch_id % len(batch_palette)],
            label=batch_name,
            s=5,
            alpha=0.72,
            linewidths=0,
            rasterized=True,
        )
    digit_ax.set_title(f"{method_label}\ndigit labels", fontsize=9, pad=5)
    batch_ax.set_title("batch labels", fontsize=9, pad=5)
    style_embedding_axis(digit_ax, embedding)
    style_embedding_axis(batch_ax, embedding)

digit_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        color=digit_cmap((digit_id + 0.5) / (digit_vmax + 0.5)),
        label=label_name,
        markersize=5,
    )
    for digit_id, label_name in enumerate(mnist_label_names)
]
batch_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        color=batch_palette[batch_id % len(batch_palette)],
        label=batch_name,
        markersize=5,
    )
    for batch_id, batch_name in enumerate(mnist_embedding_dataset_names)
]
digit_legend = legend_ax.legend(
    handles=digit_handles,
    title="digit",
    loc="upper left",
    frameon=False,
    fontsize=8,
    title_fontsize=9,
    ncols=2,
    handletextpad=0.3,
    columnspacing=0.8,
)
legend_ax.add_artist(digit_legend)
legend_ax.legend(
    handles=batch_handles,
    title="batch",
    loc="upper left",
    bbox_to_anchor=(0.0, 0.48),
    frameon=False,
    fontsize=8,
    title_fontsize=9,
)
fig.suptitle("MNIST distributed benchmark embeddings", fontsize=12)
fig;